# Chapter 4 — The BCI Loop

**Book:** *Brain–Computer Interfaces and Cognitive Computing: An Introduction*  
**Prerequisites:** Chapters 0–3.  
**Author:** Er. Rishabh Aryan

This chapter is the spine of the course. Chapters 5–7 put algorithms into one box of the diagram drawn here. Chapters 9–11 put applications around it. Chapter 12 asks who is responsible when the loop leaves the laboratory.

A demonstration that classifies archived trials is BCI *research*. An interface is a loop that a person can enter, perceive, and leave.

**Learning outcomes**

1. Draw the closed BCI loop from neural source to action and back, and name the function of each stage.
2. Distinguish open-loop analysis, closed-loop control, and shared control with operational criteria.
3. Construct a latency budget and explain which stage dominates for a named paradigm.
4. Describe co-adaptation of user and decoder as a feature of the system, not as a nuisance.
5. Specify a calibration and evaluation protocol that does not leak the future into the past.
6. Locate cognitive computing inside the loop as a source of priors and policies, not as a second interface.


---

## 4.1 From a pipeline to a loop

Begin with the linear story that vendors prefer:

$$
\text{brain} \rightarrow \text{sensor} \rightarrow \text{decode} \rightarrow \text{device}.
$$

That story is incomplete in four places.

1. The user **perceives** the device. Perception changes the next brain state. The diagram is a loop.
2. The decoder is **calibrated** on past data and **adapts**, or fails to adapt, as tissue and strategy change.
3. Many useful systems **share control** with automation: the decoder proposes, software constrains, the user vetoes.
4. Some systems **write** as well as read: stimulation is an extra path from machine to tissue.

> **Definition 4.1 (BCI loop).**  
> The closed causal path: neural activity is measured, transformed into an artificial output, delivered to the world or back to the nervous system, perceived or imposed, and thereby changes the activity that will be measured next.

> **Definition 4.2 (Open-loop analysis).**  
> Offline transformation of recorded activity into labels or reconstructions with no path by which the output can change the next sample in the same session.

Open-loop numbers can be necessary. They are not sufficient to claim an interface. Amina does not live in a cross-validation fold.


---

## 4.2 The loop, stage by stage

The following stages will be referred to by name for the rest of the book.

### Stage S0 — Neural source

Populations and, for some implants, single-unit spikes. Plasticity lives here (Chapter 2). The source is not a fixed transmitter.

### Stage S1 — Sensor and analogue front-end

Electrode, optode, or magnetometer; contact; amplification; anti-alias filter; isolation (Chapter 3). Failure here is not a machine-learning problem.

### Stage S2 — Digitisation and packaging

Sampling rate $f_s$, bit depth, packetisation, wireless telemetry if present. Telemetry latency and dropout belong in the latency budget of Section 4.4. A dropped packet is a missing measurement, not a “neutral” sample.

### Stage S3 — Preprocessing

Referencing, artefact handling, filtering, epoching or streaming windows (Chapter 6). Every filter has a group delay. A 2 Hz high-pass with a long impulse response can add tens to hundreds of milliseconds. Students who design filters only for stop-band attenuation design slow interfaces by accident.

### Stage S4 — Feature or representation

Band power, spatial projection $s_t = w^\top v_t$, covariance descriptors, learned embeddings (Chapters 6–7). This stage chooses what the decoder is allowed to see.

### Stage S5 — Decoder

A map from representation to a decision variable: a class, a character, a kinematic vector, a posterior over words, a scalar state such as workload. Formally, at step $t$,

$$
\hat{y}_t = f_\theta(z_t, h_t),
$$

where $z_t$ is the current representation, $h_t$ is optional decoder state (for example a hidden state or a language-model context), and $\theta$ are parameters last updated at calibration or by an adaptive rule.

### Stage S6 — Application interface and prior

The decoder output is almost never sent raw to a motor. A language model rescores characters. A wheelchair controller forbids cliff-directed velocity. A user interface requires dwell time before a click. This stage is where **cognitive computing** most cleanly enters: as a prior $P(y)$ or as a policy over interface modes.

### Stage S7 — Actuator, display, or stimulator

Cursor, speech synthesiser, robot, FES pulse, cortical stimulus. Each has its own dynamics. A prosthetic hand does not reach the decoded pose in one sample.

### Stage S8 — Perception, proprioception, or imposed field

The user sees the letter, hears the word, feels the robot, or receives a stimulus they did not choose. This stage closes the loop through the nervous system.

### Stage S9 — Adaptation and logging

Parameters $\theta$ update, or they do not. Sessions are logged with enough metadata to reconstruct the split that will be claimed later. If you cannot reconstruct the split, you do not have a result.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8.6, 6.4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")
ax.set_title("The BCI loop (stages S0–S9)")


def box(x, y, w, h, text, fc="#d7e4f2"):
    p = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=fc, edgecolor="#222222", linewidth=1.1,
    )
    ax.add_patch(p)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8)


box(3.4, 8.2, 3.2, 1.1, "S0 neural source", "#f3e5d8")
box(0.3, 6.2, 3.0, 1.1, "S1 sensor / front-end")
box(0.3, 4.7, 3.0, 1.1, "S2 digitise / telemetry")
box(0.3, 3.2, 3.0, 1.1, "S3 preprocess")
box(0.3, 1.7, 3.0, 1.1, "S4 features / embed")
box(3.5, 0.4, 3.0, 1.1, "S5 decoder")
box(6.7, 1.7, 3.0, 1.1, "S6 prior / policy", "#eee6f4")
box(6.7, 3.2, 3.0, 1.1, "S7 actuator / stimulus", "#e2edd8")
box(6.7, 4.7, 3.0, 1.1, "S8 perception / body")
box(6.7, 6.2, 3.0, 1.1, "S9 adapt / log", "#eee6f4")

ax.annotate("", xy=(3.4, 8.6), xytext=(1.8, 7.3),
            arrowprops=dict(arrowstyle="->", color="#333333"))
ax.annotate("", xy=(1.8, 6.2), xytext=(1.8, 5.8),
            arrowprops=dict(arrowstyle="-", color="#333333"))
ax.annotate("", xy=(5.0, 8.2), xytext=(8.2, 7.3),
            arrowprops=dict(arrowstyle="->", color="#333333"))
ax.annotate("", xy=(8.2, 6.2), xytext=(8.2, 5.8),
            arrowprops=dict(arrowstyle="-", color="#333333"))
ax.text(5.0, 9.55, "tissue and strategy change S0 over time", ha="center", fontsize=8, color="#444444")
plt.tight_layout()
plt.show()


Read the figure clockwise from S0 down the left column and up the right. Cognitive computing is painted on S6 and S9 on purpose. It is not painted on S0.


---

## 4.3 Control modes

### Open loop

Labels are assigned after the fact. The user may have been performing a task, but the output of $f_\theta$ did not steer the next stimulus. Use: method development, ablation studies, sanity checks. Abuse: quoting open-loop accuracy as if Amina had communicated.

### Closed loop, user in command

$\hat{y}_t$ moves a cursor, selects a letter, or speaks a word, and the user sees the result in time to correct. This is the default meaning of “interface” in Chapters 9 and 10.

### Shared control

The decoder output is a *proposal*. Automation rejects physically impossible commands, completes a grasp, or snaps to a vocabulary item. Shared control is not cheating if it is disclosed. It is cheating if a paper reports the success of the compound system as if it were the success of the neural decoder alone.

> **Definition 4.3 (Shared control).**  
> A policy that combines a neural decision variable with non-neural constraints or assistance before the actuator is moved.

### Passive loop

No command is issued *as a command*. Stage S5 estimates a state (workload, fatigue, affect). Stage S6 changes the environment: slows the matrix, pauses the lesson, warns a supervisor. The user may not have agreed to that estimate being used. Chapter 11 treats the workplace case.

### Bidirectional loop

Stage S7 includes a stimulator whose parameters depend on $\hat{y}_t$ or on a detected state. Artefact from S7 into S1 is now a first-class engineering object (Chapter 3).


---

## 4.4 Time: latency, rate, and stability

An interface that is accurate and late is inaccurate in practice. Three clocks must be kept separate.

**Sample clock.** $f_s$ from Stage S2. Sets the finest temporal grain of the digital stream.

**Update clock.** How often $\hat{y}_t$ is emitted. A P300 system may emit one decision after a sequence of flashes lasting seconds. A cursor system may emit a velocity every 20–50 ms.

**User clock.** How long Amina will tolerate before the world answers. Conversation-like speech is unforgiving. A letter every few seconds may still beat silence.

> **Definition 4.4 (Loop latency).**  
> The time from a neural event that *should* change the output to the first perceptible change in Stage S7 or S8.

A first latency budget, to be filled with measured numbers rather than hopes:

| Stage | Typical contributors to delay |
|---|---|
| S1–S2 | Analogue filtering; packetisation; radio |
| S3 | Digital filter group delay; artefact buffers |
| S4–S5 | Window length; model compute |
| S6 | Language-model context; UI dwell |
| S7 | Motor dynamics; speech-synthesis buffer |
| S8 | Human reaction and display refresh |

Window length is the term students underestimate. A 1-second FFT window has already spent 1 second before the decoder speaks, plus hop size if you overlap. Claiming “real time” with a 2-second causal window is a contradiction in terms.

**Information rate is not loop latency.** A system can be slow per decision and still high-rate if each decision carries many bits. A system can be fast and useless if each decision is a coin toss. Chapter 9 will compute information transfer rate and then explain why the number misleads. Here it is enough to refuse a single figure of merit.


In [ ]:
import numpy as np

# Causal sliding-window latency cartoon.
fs = 256
window_s = 1.00
hop_s = 0.10
filter_delay_s = 0.040
compute_s = 0.015
ui_dwell_s = 0.000  # continuous cursor; dwell would add more
radio_s = 0.020

window_delay = window_s  # causal window: must wait for the last sample
loop_ms = 1000 * (window_delay + hop_s + filter_delay_s + compute_s + ui_dwell_s + radio_s)

print("causal window (s)     ", window_s)
print("hop (s)               ", hop_s)
print("filter group delay (s)", filter_delay_s)
print("compute (s)           ", compute_s)
print("radio / packeting (s) ", radio_s)
print("approx loop latency   ", f"{loop_ms:.0f} ms")
print("Note: the window dominates. Shortening the model without shortening")
print("the window will not make this interface feel faster.")


Run the cell again with `window_s = 0.25` and watch the budget collapse. Then ask whether the feature you wanted still exists in 250 ms. That question *is* the design.


---

## 4.5 The user is a plant and a controller

Control theory would call the nervous system the plant. That vocabulary is incomplete. The user is also a controller who invents strategies.

Examples of strategy, all documented in the literature in some form:

- tensing a forehead muscle until an EEG class flips (artefact as policy);
- delaying imagery until a useful phase of a flicker;
- abandoning a letter and restarting a word because the language model is stubborn;
- reducing effort once the shared-control layer can be trusted.

> **Definition 4.5 (Co-adaptation).**  
> Joint change of user strategy and decoder parameters over the life of the loop.

Co-adaptation can raise performance. It can also hide a decoder that has learned a non-neural shortcut. Two experimental obligations follow.

1. **Log strategy.** After a session, ask what the user did. Write it down. A method section that omits this is incomplete.
2. **Audit shortcuts.** Inspect whether EMG, eye movement, or timing artefacts predict $\hat{y}_t$ better than the claimed neural feature (Chapter 6). If they do, the loop is real and the scientific claim is false.

Cognitive models of workload and fatigue belong in Stage S6 as *policies for protecting the user*, not as romantic descriptions of S0.


---

## 4.6 Calibration, adaptation, and the life of $\theta$

Parameters $\theta$ do not fall from theory. They are estimated.

**Session calibration.** A structured task with known labels produces a first $\theta$. Cost: Amina’s morning. A method that needs forty minutes of calibration to deliver five minutes of use has not understood Stage S8.

**Recalibration.** $\theta$ is refreshed when drift is detected or on a schedule. Detecting drift without labels is itself a research problem; do not pretend it is solved because a loss curve looked stable offline.

**Continuous adaptation.** $\theta$ updates from incoming data. If updates use decoder outputs as labels, errors can reinforce themselves. If updates use an explicit “undo” key, the undo key is part of the interface and must be reported.

**Subject-independent models.** $\theta$ is estimated on other people. Chapter 7 will treat this as a domain-shift problem. It does not abolish calibration; it changes how much of $\theta$ is local.

> **Definition 4.6 (Calibration burden).**  
> The user’s time, fatigue, and setup labour required to obtain a $\theta$ that meets a pre-declared performance criterion on held-out *later* data.

State the criterion before you see the number. Otherwise you will move the criterion.


---

## 4.7 Evaluation inside the loop

Chapter 0 showed that a shuffled split leaks. The loop adds further ways to lie.

**Offline accuracy on the calibration set.** Not a result.

**Offline accuracy on a shuffled partition of one session.** Not a claim about tomorrow.

**Closed-loop success with an expert operator holding the cap.** A result about that operator.

**Closed-loop success with Amina, at home, after week four, with calibration time counted against the day.** A result about an interface.

Metrics that belong in a loop paper or a loop chapter of a thesis:

- task success the user will accept (letters per minute *after* corrections; words the listener understands);
- loop latency, measured, not hoped;
- calibration burden (Definition 4.6);
- dropout and packet loss;
- time to recover after an artefact or an undo;
- how often shared control, not the decoder, saved the trial;
- whether the user would choose the system again on Tuesday.

Chance level must be stated beside every percentage. A two-class balanced task has chance $0.5$. A 36-character matrix does not.


In [ ]:
# Chance level is not optional.
def chance_two_class_balanced():
    return 0.5


def chance_character_matrix(n_chars=36):
    return 1.0 / n_chars


def letters_per_min(correct_chars, minutes, including_undo=True):
    # including_undo=True means failed attempts consumed time already in minutes.
    return correct_chars / minutes


print("chance, 2-class        ", chance_two_class_balanced())
print("chance, 6x6 matrix     ", round(chance_character_matrix(36), 4))
print("12 correct letters in 4 min, undos included in the 4 min:")
print("  letters / min        ", letters_per_min(12, 4))


---

## 4.8 Worked loops

### Loop A — Reactive P300 communication (Amina, EEG)

S0: attentional enhancement of a rare flash. S1–S2: wet or dry EEG. S3: epoching locked to flashes. S4: time-domain or spatial-filter features of the ERP. S5: classifier scores flashes. S6: accumulation over repetitions plus optional language model. S7: character appears. S8: Amina reads it and decides whether to undo. Latency is dominated by repetitions, not by the classifier. The honest figure of merit is corrected letters per minute and setup time, not trial-wise AUC.

### Loop B — Active motor imagery cursor

S0: modulation of sensorimotor rhythms. S4: band power after a spatial filter. S5: continuous score mapped to velocity. S6: smoothing and maybe shared-control snapping. S7: cursor. S8: vision. Co-adaptation is intense. Artefact audit is mandatory.

### Loop C — Attempted-speech implant

S0: population activity in speech-related cortex. S4–S5: sequence model over short windows. S6: language model prior over words. S7: text or synthetic voice. S8: Amina hears herself and continues or repairs. Latency is now a conversational constraint. Shared control by the language model must be disclosed or the decoder will be credited with English.

### Loop D — Passive workload in training software

S5 outputs a scalar. S6 changes task difficulty. No command. The ethical object is the scalar leaving the room. That is a loop with a supervisor in it, whether or not the methods section says so.


---

## 4.9 Failure modes of the loop

Keep this list beside any design document.

1. **Broken contact (S1).** The decoder classifies impedance.
2. **Destroyed band (S2–S3).** Sampling or filtering removed the feature you named.
3. **Non-causal evaluation.** Tomorrow leaked into $\theta$.
4. **Shortcut.** EMG or eye-track predicts better than the neural feature.
5. **Uncounted shared control.** The robot finished the grasp.
6. **Uncounted prior.** The language model wrote the sentence.
7. **Intolerable latency.** Accurate after the conversation has moved on.
8. **Calibration that consumes the day.**
9. **Silent drift.** $\theta$ is frozen; tissue is not.
10. **Abandoned loop.** Stage S9 has no owner in year five (Definition 3.4).

A paper that reports only item-free accuracy has not excluded these.


---

## 4.10 Where cognitive computing sits — and does not

Re-state Chapter 1 with the loop as witness.

- **Priors at S6.** Language models, task grammars, road maps for a wheelchair.
- **Policies at S6 and S9.** Slow the matrix when inferred fatigue is high; request recalibration when drift is inferred; refuse to issue a workplace score the contract did not name.
- **Models of the user as controller.** Limited working memory, accumulator decisions, attentional capacity — used to *design* trial pace, not to claim that S5 has recovered the content of a thought.
- **Neuromorphic hardware.** A possible implementation of S3–S5 near the sensor. It does not move the system off the resolution–risk surface of Chapter 3.

A sentence that says “cognitive computing closes the loop” is incomplete until it names the stage.


---

## 4.11 A design sheet you can reuse

Copy this sheet into laboratory notes before you collect a session.

1. User and setting (name the constraints).  
2. Purpose theme and active / reactive / passive (Chapter 1).  
3. Modality and point on the resolution–risk surface (Chapter 3).  
4. Stages S0–S9 in one paragraph each.  
5. Control mode (open, closed, shared, passive, bidirectional).  
6. Latency budget with measured or target numbers.  
7. What $\theta$ is, how it is obtained, how it is updated.  
8. What the prior or policy at S6 is allowed to do.  
9. Evaluation: split, chance level, user-facing metric, artefact audit.  
10. Owner of the loop after the session (especially if implanted).

If a line cannot be filled, the study is not ready to run.


---

## 4.12 What to carry into Chapter 5

Chapter 5 fills Stage S0–S5 with paradigms that already work. Carry five sentences.

1. An interface is a loop; an offline score is a tool.
2. Shared control and linguistic priors are part of the system; credit them or do not claim the remainder.
3. Window length often dominates latency.
4. The user changes S0 on purpose.
5. If you cannot name the split and the owner, you cannot name the result.


---

## 4.13 Chapter summary

The BCI loop runs from a plastic neural source through sensor, digitisation, preprocessing, representation, decoder, prior or policy, actuator or stimulator, and perception, and back. Open-loop analysis supports method work. Closed-loop, shared-control, passive, and bidirectional modes change both the engineering and the ethics. Latency is a budget dominated by windows and repetitions more often than by model size. Co-adaptation is expected. Evaluation that cannot survive a later block, an artefact audit, and a user-facing metric has not evaluated an interface. Cognitive computing belongs at the prior and the policy, not as a synonym for the loop.


---

## Exercises

**4.1** Redraw Stages S0–S9 for Amina using a P300 speller. Write one sentence per stage. Mark where a language model, if present, sits.

**4.2** A paper reports 94 percent offline accuracy for motor imagery, random 5-fold on windows cut with 80 percent overlap from one session. Which failure modes from Section 4.9 apply? Which metric would you demand instead?

**4.3** Using the latency cell, compute loop latency for window lengths 0.25, 0.50, and 1.00 s with hop one-tenth of the window, filter delay 40 ms, compute 15 ms, radio 20 ms. For each, state whether a 2 Hz-wide mu-power feature is even well-defined.

**4.4** Define shared control in your own words. Then give one example in which shared control is ethically required (safety) and one in which it inflates a scientific claim.

**4.5** Amina’s morning calibration lasts 35 minutes and yields a decoder that works for 12 minutes before drift. Using Definition 4.6, argue whether the loop is finished.

**4.6** Write a six-line artefact-audit protocol for an EEG cursor study that would convince a sceptical examiner the class is not forehead EMG.

**4.7** Place a passive workload estimator in S0–S9. Who, besides the user, is in this loop if the software is an employer’s training system?

**4.8** Using only this chapter, explain in 250 words why “real-time deep learning BCI” is not a design. What would you need to add to make it one?


---

## Annotated reading

- Wolpaw and Wolpaw, *Brain–Computer Interfaces: Principles and Practice* — chapters on the closed loop and on adaptive control; use as the encyclopaedic counterpart of this spine.  
- Fabien Lotte and colleagues, tutorials on designing a BCI end-to-end — for the checklist spirit of Section 4.11.  
- Work on co-adaptation and BCI illiteracy — read after you can name S0 and S5 separately.  
- A paper that reports closed-loop speech or cursor use *at home*; compare its metrics with Section 4.7.

---

*End of Chapter 4.*  
*Next: Chapter 5 — Paradigms that already work.*
